# Paper Figures: Figure 4 - Data Aligned to Transitions

This notebook generates Figure 4 showing behavioural and dopamine (photometry) responses aligned to the transition point where animals shift from high to low appetitive behaviour


In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path("../src").resolve()))
from data_io import load_assembled

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib.colors import to_hex

from trompy import save_figure_atomic

from figure_config import (
    configure_matplotlib, COLORS, HEATMAP_CMAP_DIV,
    DATAFOLDER, RESULTSFOLDER, FIGSFOLDER,
    SAVE_FIGS
)

from figure_plotting import (plot_auc_and_sigmoid, plot_realigned_behaviour,
                             make_realignment_panel_behav_and_da,
                             make_realignment_panel_only_one_parameter,
                             make_realignment_slope_panel
)

from model_fit_helpers import (
    fit_sigmoid,
    _sigmoid_model,
)

# from utils import make_realigned_trials
from realignment_helpers import get_realigned_data

SAVE_FIGS = True
FIT_TO_RAW_DATA = True

# Configure matplotlib
configure_matplotlib()
colors = COLORS  # Use shared color palette
custom_cmap = HEATMAP_CMAP_DIV  # Use shared colormap

# 11 evenly spaced colors from custom_cmap
sampled_hex = [to_hex(custom_cmap(x)) for x in np.linspace(0, 1, 11)]
DA_COLOR = sampled_hex[0]  # Choose the 3rd color for DA
BEHAV_COLOR = sampled_hex[-1]  # Choose the 6th color for behavior

## Load Assembled Data
Load the complete dataset with transition points and trial-aligned indices.

In [ ]:
assembled_data_path = DATAFOLDER / "assembled"

data = load_assembled(assembled_data_path)

# Extract main components
x_array = data["x_array"].copy()

# reverses cluster assignments
x_array["cluster_photo"] = x_array["cluster_photo"].map({0: 1, 1: 0})

print("Loaded assembled data from data/assembled")
print(f"\nData structure:")
print(f"  - x_array shape: {x_array.shape}")

## Realignment figure

In [ ]:
## prepare shared data
from realignment_helpers import only_keep_complete_trials

def prepare_shared_data(df, realignment_col, da_col, simba_col, initial_da_k=-1, initial_behav_k=-1, ntrials_for_slope=5, offset=0):
    """Compute shared fits/arrays used by all Figure 4B panels."""

    df_realigned = only_keep_complete_trials(df, realignment_col).copy()
    df_reali_for_slope = df_realigned.query(f"{realignment_col} >= -{ntrials_for_slope}+{offset} and {realignment_col} <= {ntrials_for_slope}+{offset}").copy()
    
    ntrials = df_realigned[realignment_col].nunique()
    midway_point = int(df[df[realignment_col] == 0].trial.mean())
    min_trial, max_trial = midway_point - ntrials // 2, midway_point + ntrials // 2
    
    print(midway_point, min_trial, max_trial)
    
    df_original = df.query("trial >= @min_trial and trial <= @max_trial").copy()
    
    min_trial, max_trial = midway_point - ntrials_for_slope, midway_point + ntrials_for_slope
    df_orig_for_slope = df_original.query("trial >= @min_trial and trial <= @max_trial").copy()

    # popt_behav_orig = fit_sigmoid(df, "trial", simba_col, fit_to_raw_data=FIT_TO_RAW_DATA, initial_k=initial_behav_k)
    # popt_behav_realigned = fit_sigmoid(df_realigned, realignment_col, simba_col, fit_to_raw_data=FIT_TO_RAW_DATA, initial_k=initial_behav_k)
    # popt_da_orig = fit_sigmoid(df, "trial", da_col, fit_to_raw_data=FIT_TO_RAW_DATA, initial_k=initial_da_k)
    # popt_da_realigned = fit_sigmoid(df_realigned, realignment_col, da_col, fit_to_raw_data=FIT_TO_RAW_DATA, initial_k=initial_da_k)

    # k_behav_orig = popt_behav_orig[2] if np.all(np.isfinite(popt_behav_orig)) else np.nan
    # k_behav_realigned = popt_behav_realigned[2] if np.all(np.isfinite(popt_behav_realigned)) else np.nan
    # k_da_orig = popt_da_orig[2] if np.all(np.isfinite(popt_da_orig)) else np.nan
    # k_da_realigned = popt_da_realigned[2] if np.all(np.isfinite(popt_da_realigned)) else np.nan

    return {
        "df_original": df_original,
        "df_orig_for_slope": df_orig_for_slope,
        "df_realigned": df_realigned,
        "df_reali_for_slope": df_reali_for_slope,
        # "k_behav_orig": k_behav_orig,
        # "k_behav_realigned": k_behav_realigned,
        # "k_da_orig": k_da_orig,
        # "k_da_realigned": k_da_realigned,
        # "popt_behav_orig": popt_behav_orig,
        # "popt_behav_realigned": popt_behav_realigned,
        # "popt_da_orig": popt_da_orig,
        # "popt_da_realigned": popt_da_realigned,
        "ntrials_for_slope": ntrials_for_slope,
    }



In [ ]:
# new figure 4

# prep data for realignment panels
DA_COLUMN = "auc_snips"
SIMBA_COLUMN = "simba_median_balance"

RATS_TO_EXCLUDE = ["PB27", # This rat has a v poor fits of behavioral data
                   #"PB71" # This rat is on the edge, changes behavior and then goes into passive drip mode
                   ] 

# realigning behavior to da
REALIGNMENT_COLUMN_TO_DA = "realigned_trials_da"
OFFSET = 3
subset_aligned_to_da = get_realigned_data(x_array, REALIGNMENT_COLUMN_TO_DA, rats_to_exclude=RATS_TO_EXCLUDE, verbose=False)
data_for_figure_to_da = prepare_shared_data(subset_aligned_to_da, REALIGNMENT_COLUMN_TO_DA, DA_COLUMN, SIMBA_COLUMN, ntrials_for_slope=5, offset=OFFSET)

fig, ax = make_realignment_panel_only_one_parameter(data_for_figure_to_da, REALIGNMENT_COLUMN_TO_DA, SIMBA_COLUMN, "behav",
                                          color=BEHAV_COLOR, do_not_plot_sigmoid=True)

ax[0].axvline(17, color="gray", linestyle="--", alpha=0.5)
ax[1].axvline(0, color="gray", linestyle="--", alpha=0.5)

for axis in ax:
    axis.set_ylim([-0.5, 1])

save_figure_atomic(fig, "fig4_realignment_behaviour", FIGSFOLDER)

# realigning da to behav

REALIGNMENT_COLUMN_TO_BEHAV = "realigned_trials_behav"
OFFSET = -3
subset_aligned_to_behav = get_realigned_data(x_array, REALIGNMENT_COLUMN_TO_BEHAV, rats_to_exclude=RATS_TO_EXCLUDE, verbose=False)
data_for_figure_to_behav = prepare_shared_data(subset_aligned_to_behav, REALIGNMENT_COLUMN_TO_BEHAV, DA_COLUMN, SIMBA_COLUMN, ntrials_for_slope=5, offset=OFFSET)

fig, ax = make_realignment_panel_only_one_parameter(data_for_figure_to_behav, REALIGNMENT_COLUMN_TO_BEHAV, DA_COLUMN, "da",
                                          color=DA_COLOR, do_not_plot_sigmoid=True)

ax[0].axvline(18, color="gray", linestyle=":", alpha=0.5)
ax[1].axvline(0, color="gray", linestyle=":", alpha=0.5)

print(ax[1].get_ylim())
save_figure_atomic(fig, "fig4_realignment_da", FIGSFOLDER)

In [ ]:
# make slope panels

f, ax, slopes =make_realignment_slope_panel(data_for_figure_to_da, SIMBA_COLUMN, REALIGNMENT_COLUMN_TO_DA,
                             color=BEHAV_COLOR)

ax.set_yticks([ -0.5, 0, 0.5, 1])
ax.set_ylabel("Valence index")
ax.set_xticks([-5, 0, 5], ["-2", "+3", "+8"])
ax.axvline(-3, color="gray", linestyle="--", alpha=0.5)

save_figure_atomic(f, "fig4_realignment_slope_behav", FIGSFOLDER)


print("Behaviour slopes")
orig = slopes["trial"]
print("Orig. slope = {: .3f}".format(orig["slope"]))
print("Orig. r = {: .3f}".format(orig["r"]))
print("Orig. p-value (H0: slope = 0) = {:.4g}".format(orig["p"]))
print("Orig. significant vs 0:", "yes" if orig["p"] < 0.05 else "no")
print("Reali. slope = {: .3f}".format(slopes["realigned_trials_da"]["slope"]))
print("Reali. r = {: .3f}".format(slopes["realigned_trials_da"]["r"]))
print("Reali. p-value (H0: slope = 0) = {:.4g}".format(slopes["realigned_trials_da"]["p"]))
print("Reali. significant vs 0:", "yes" if slopes["realigned_trials_da"]["p"] < 0.05 else "no")

f, ax, slopes =make_realignment_slope_panel(data_for_figure_to_behav, DA_COLUMN, REALIGNMENT_COLUMN_TO_BEHAV,
                             color=DA_COLOR)


# ax.set_yticks([-0.5, 0, 0.5, 1])
ax.set_ylabel("Dopamine (AUC)")
ax.set_xticks([-5, 0, 5], ["-8", "-3", "+2"])
ax.axvline(3, color="gray", linestyle=":", alpha=0.5)
ax.set_ylim([np.float64(-8.289015319171442), np.float64(15.032973623363642)])

save_figure_atomic(f, "fig4_realignment_slope_da", FIGSFOLDER)

print("Dopamine slopes")
print("Orig. slope = {: .3f}".format(slopes["trial"]["slope"]))
print("Orig. r = {: .3f}".format(slopes["trial"]["r"]))
print("Orig. p-value (H0: slope = 0) = {:.4g}".format(slopes["trial"]["p"]))
print("Orig. significant vs 0:", "yes" if slopes["trial"]["p"] < 0.05 else "no")

print("Reali. slope = {: .3f}".format(slopes["realigned_trials_behav"]["slope"]))
print("Reali. r = {: .3f}".format(slopes["realigned_trials_behav"]["r"]))
print("Reali. p-value (H0: slope = 0) = {:.4g}".format(slopes["realigned_trials_behav"]["p"]))
print("Reali. significant vs 0:", "yes" if slopes["realigned_trials_behav"]["p"] < 0.05 else "no")

In [ ]:
from bootstrap_and_shuffle_helpers import get_bootstrapped_distribution, get_bootstrapped_distribution_using_slopes, get_bootstrapped_distribution_using_r

N_BOOTSTRAPS = 1000
BASE_BOOTSTRAP_SEED = 42

# can change these for quick testing but make sure to comment again to ensure that values from plots above are carried through to the bootstrapped distributions
# DA_COLUMN = "auc_snips"
# SIMBA_COLUMN = "simba_median_balance"
# REALIGNMENT_COLUMN = "realigned_trials_behav"
# subset_aligned = get_realigned_data(x_array, REALIGNMENT_COLUMN, rats_to_exclude=RATS_TO_EXCLUDE, verbose=False)
# data_for_figure = prepare_shared_data(subset_aligned, REALIGNMENT_COLUMN, DA_COLUMN, SIMBA_COLUMN, ntrials_for_slope=5)

# Use Pearson r for normalized cross-parameter comparison (behavior vs dopamine).
df_orig_for_slope = data_for_figure_to_da["df_orig_for_slope"]
df_reali_for_slope = data_for_figure_to_da["df_reali_for_slope"]

bootstrapped_behav_orig = get_bootstrapped_distribution_using_slopes(
    df_orig_for_slope,
    "trial",
    SIMBA_COLUMN,
    n_bootstraps=N_BOOTSTRAPS,
    random_seed=BASE_BOOTSTRAP_SEED + 0,
)
bootstrapped_behav_realigned = get_bootstrapped_distribution_using_slopes(
    df_reali_for_slope,
    REALIGNMENT_COLUMN_TO_DA,
    SIMBA_COLUMN,
    n_bootstraps=N_BOOTSTRAPS,
    random_seed=BASE_BOOTSTRAP_SEED + 1,
)

df_orig_for_slope = data_for_figure_to_behav["df_orig_for_slope"]
df_reali_for_slope = data_for_figure_to_behav["df_reali_for_slope"]

bootstrapped_da_orig = get_bootstrapped_distribution_using_slopes(
    df_orig_for_slope,
    "trial",
    DA_COLUMN,
    n_bootstraps=N_BOOTSTRAPS,
    random_seed=BASE_BOOTSTRAP_SEED + 2,
)
bootstrapped_da_realigned = get_bootstrapped_distribution_using_slopes(
    df_reali_for_slope,
    REALIGNMENT_COLUMN_TO_BEHAV,
    DA_COLUMN,
    n_bootstraps=N_BOOTSTRAPS,
    random_seed=BASE_BOOTSTRAP_SEED + 3,
)

In [ ]:
from figure_plotting import make_bootstrap_boxplot

f, ax = make_bootstrap_boxplot([bootstrapped_behav_orig, bootstrapped_behav_realigned], BEHAV_COLOR)
ax.set_ylabel("Slope")
ax.set_yticks([-0.1, 0, 0.1])
# ax.set_ylim(-0.8, 0.5)

save_figure_atomic(f, "fig4_realignment_bootstrap_behaviour", FIGSFOLDER)

f, ax = make_bootstrap_boxplot([bootstrapped_da_orig, bootstrapped_da_realigned], DA_COLOR)
ax.set_ylabel("Slope")
ax.set_yticks([-2,-1, 0])
# ax.set_ylim(-0.8, 0.5)

save_figure_atomic(f, "fig4_realignment_bootstrap_da", FIGSFOLDER)

In [ ]:
# Paired bootstrap-difference inference (realigned - original)
def paired_bootstrap_diff_stats(orig, realigned, label, ci=(2.5, 97.5), alternative="greater"):
    orig = np.asarray(orig, dtype=float)
    realigned = np.asarray(realigned, dtype=float)

    if orig.shape != realigned.shape:
        n = min(orig.size, realigned.size)
        print(f"{label}: length mismatch (orig={orig.size}, realigned={realigned.size}); trimming to n={n}")
        orig = orig[:n]
        realigned = realigned[:n]

    valid = np.isfinite(orig) & np.isfinite(realigned)
    deltas = realigned[valid] - orig[valid]
    n = deltas.size

    if n == 0:
        print(f"{label}: no valid paired bootstrap samples")
        return np.array([])

    delta_mean = np.mean(deltas)
    ci_low, ci_high = np.percentile(deltas, ci)

    # One-sided bootstrap p-value for H0: delta == 0
    # alternative='greater' tests H1: delta > 0; 'less' tests H1: delta < 0
    if alternative == "greater":
        p_one_sided = (np.sum(deltas <= 0) + 1) / (n + 1)
    elif alternative == "less":
        p_one_sided = (np.sum(deltas >= 0) + 1) / (n + 1)
    else:
        raise ValueError("alternative must be 'greater' or 'less'")

    print(f"{label} (paired bootstrap delta = realigned - original)")
    print(f"  n valid pairs = {n}")
    print(f"  mean delta = {delta_mean:.6f}")
    print(f"  95% CI = [{ci_low:.6f}, {ci_high:.6f}]")
    print(f"  one-sided bootstrap p ({alternative}) = {p_one_sided:.6f}")

    return deltas


delta_da = paired_bootstrap_diff_stats(
    bootstrapped_da_orig,
    bootstrapped_da_realigned,
    label="Dopamine",
    alternative="less",
)

delta_behav = paired_bootstrap_diff_stats(
    bootstrapped_behav_orig,
    bootstrapped_behav_realigned,
    label="Behavior",
    alternative="less",
)

print(f"Paired bootstrap-difference test revealed behavior slopes were marginally smaller "
      f"in realigned versus original data (Δ = −0.055, 95% CI [−0.122, 0.005], p = 0.039, "
      f"n = 1000 bootstrap resamples)")

In [ ]:
from scipy.stats import ttest_ind, ttest_1samp, ttest_rel
# Dopamine vs 0
ttest_1samp(bootstrapped_da_realigned, 0)

print(f"Behaviour (orig vs realigned): {ttest_ind(bootstrapped_behav_orig, bootstrapped_behav_realigned)}")
print(f"Dopamine (orig vs realigned): {ttest_ind(bootstrapped_da_orig, bootstrapped_da_realigned)}")

In [ ]:
from scipy.stats import mannwhitneyu
stat, p = mannwhitneyu(bootstrapped_behav_orig, bootstrapped_behav_realigned)
print(f"Mann-Whitney U: U={stat:.1f}, p={p:.4f}")

ci_orig = np.percentile(bootstrapped_behav_orig, [2.5, 97.5])
ci_real = np.percentile(bootstrapped_behav_realigned, [2.5, 97.5])
print(f"Original: 95% CI = [{ci_orig[0]:.4f}, {ci_orig[1]:.4f}]")
print(f"Realigned: 95% CI = [{ci_real[0]:.4f}, {ci_real[1]:.4f}]")
# If CIs don't overlap → distributions differ

In [ ]:
BANDWIDTH=1

fig, ax = plt.subplots(figsize=(2.3, 2.1),
                     gridspec_kw={"left": 0.25, "right": 0.9, "top": 0.8, "bottom": 0.25})

sns.kdeplot(bootstrapped_behav_orig, color="grey", ax=ax, cut=0, fill=True, bw_adjust=BANDWIDTH)
sns.kdeplot(bootstrapped_behav_realigned, color=BEHAV_COLOR, ax=ax, cut=0, fill=True, bw_adjust=BANDWIDTH)

ax.set_xlabel("Pearson r")
ax.set_xticks([-1, -0.5, 0, 0.5, 1])
ax.set_xlim(-1, 1)
sns.despine(ax=ax, offset=5)

save_figure_atomic(fig, FIGSFOLDER / "fig3_realigned_bootstrap_kde_behaviour")

fig, ax = plt.subplots(figsize=(2.3, 2.1),
                     gridspec_kw={"left": 0.25, "right": 0.9, "top": 0.8, "bottom": 0.25})

sns.kdeplot(bootstrapped_da_orig, color="grey", ax=ax, cut=0, fill=True, bw_adjust=BANDWIDTH)
sns.kdeplot(bootstrapped_da_realigned, color=DA_COLOR, ax=ax, cut=0, fill=True, bw_adjust=BANDWIDTH)

ax.set_xlabel("Pearson r")
ax.set_xticks([-1, -0.5, 0, 0.5, 1])
ax.set_xlim(-1, 1)
sns.despine(ax=ax, offset=5)

save_figure_atomic(fig, FIGSFOLDER / "fig3_realigned_bootstrap_kde_dopamine")